# 🧪 Cartographer — Eval Runner

Runs all curated test cases through the full Cartographer graph and scores them
using an LLM-as-judge rubric across 5 dimensions.

## Rubric
| Dimension | Weight | Description |
|---|---|---|
| Breadth | 25% | All expected topics covered? |
| Depth | 30% | Surface-level vs. nuanced insights |
| Recency | 20% | Sources appear recent |
| Source Diversity | 15% | Multiple distinct sources/perspectives |
| Citation Quality | 10% | Reputable, accessible URLs |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

from dotenv import load_dotenv
load_dotenv('../.env')

import json
import asyncio
from datetime import datetime
import pandas as pd
from IPython.display import display, Markdown

from src.graph import cartographer
from src.llm_factory import build_llm
from langchain_core.messages import HumanMessage, SystemMessage

print('✅ Setup complete')

In [ ]:
# Load test cases
with open('test_cases.json') as f:
    TEST_CASES = json.load(f)

print(f'Loaded {len(TEST_CASES)} test cases:')
for tc in TEST_CASES:
    print(f'  [{tc["id"]}] {tc["quest"][:60]}…')

In [ ]:
JUDGE_SYSTEM = """
You are an expert research evaluator. Score the provided research report against the original quest.

Evaluate on these 5 dimensions (return JSON only):
{
  "breadth": <0-10>,       // All expected topics covered?
  "depth": <0-10>,         // Nuanced, detailed, not surface-level?
  "recency": <0-10>,       // Sources appear recent and current?
  "source_diversity": <0-10>, // Multiple perspectives / source types?
  "citation_quality": <0-10>, // URLs look credible and varied?
  "topics_covered": ["list", "of", "covered", "topics"],
  "topics_missing": ["list", "of", "missing", "topics"],
  "reasoning": "<1-2 sentence overall assessment>"
}

Return ONLY the JSON. No markdown, no explanation.
"""

async def judge_report(quest: str, expected_topics: list, report: str, sources: list) -> dict:
    """LLM-as-judge: scores a report against expected topics."""
    judge_llm = build_llm(streaming=False, temperature=0.1)
    
    source_list = '\n'.join(f'  - {s["url"]}' for s in sources[:10])
    
    messages = [
        SystemMessage(content=JUDGE_SYSTEM),
        HumanMessage(content=(
            f"Quest: {quest}\n\n"
            f"Expected topics: {json.dumps(expected_topics)}\n\n"
            f"Sources used ({len(sources)} total):\n{source_list}\n\n"
            f"Report (first 3000 chars):\n{report[:3000]}"
        )),
    ]
    
    response = await judge_llm.ainvoke(messages)
    raw = response.content.strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
        raw = raw.strip()
    
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'error': 'parse_failed', 'raw': raw}


def weighted_score(scores: dict) -> float:
    """Apply rubric weights to dimension scores."""
    weights = {
        'breadth': 0.25,
        'depth': 0.30,
        'recency': 0.20,
        'source_diversity': 0.15,
        'citation_quality': 0.10,
    }
    return sum(scores.get(dim, 0) * w for dim, w in weights.items())


print('✅ Judge functions ready')

In [ ]:
# ── Run all test cases ────────────────────────────────────────────────────────
# ⚠️  This will consume API credits (Tavily + LLM). Run selectively if needed.

results = []

for tc in TEST_CASES:
    print(f"\n{'='*60}")
    print(f"Running [{tc['id']}]: {tc['quest'][:55]}…")
    print(f"{'='*60}")
    
    initial_state = {
        'quest': tc['quest'],
        'waypoints': [], 'terrain': [], 'uncharted_zones': [],
        'coverage_score': 0.0, 'expedition_count': 0,
        'treasure_map': '', 'sources': [], 'trace': [],
    }
    
    # Run graph
    final = None
    async for event in cartographer.astream_events(initial_state, version='v2'):
        if event['event'] == 'on_chain_end' and event.get('name') == 'LangGraph':
            final = event['data']['output']
        elif event['event'] == 'on_chain_end' and event.get('name') in ('planner','explorer','critic','writer'):
            for t in event['data'].get('output', {}).get('trace', []):
                print(f"  {t['message'][:80]}")
    
    if not final:
        print(f"  ❌ Graph failed for {tc['id']}")
        continue
    
    # Judge the report
    judge_scores = await judge_report(
        tc['quest'], tc['expected_topics'],
        final.get('treasure_map', ''),
        final.get('sources', [])
    )
    
    ws = weighted_score(judge_scores)
    passed = (
        ws >= tc['min_coverage_score'] and
        len(final.get('sources', [])) >= tc['min_sources']
    )
    
    results.append({
        'id': tc['id'],
        'quest': tc['quest'][:50] + '…',
        'domain': tc['domain'],
        'weighted_score': round(ws, 2),
        'breadth': judge_scores.get('breadth', 0),
        'depth': judge_scores.get('depth', 0),
        'recency': judge_scores.get('recency', 0),
        'source_diversity': judge_scores.get('source_diversity', 0),
        'citation_quality': judge_scores.get('citation_quality', 0),
        'sources_count': len(final.get('sources', [])),
        'min_sources': tc['min_sources'],
        'expeditions': final.get('expedition_count', 0),
        'passed': '✅' if passed else '❌',
        'reasoning': judge_scores.get('reasoning', ''),
    })
    
    print(f"  Score: {ws:.2f}/10 | Sources: {len(final.get('sources',[]))} | {'PASS ✅' if passed else 'FAIL ❌'}")

print(f"\n✅ Eval complete: {len(results)}/{len(TEST_CASES)} cases ran")

In [ ]:
# ── Results Table ─────────────────────────────────────────────────────────────
df = pd.DataFrame(results)

display(Markdown('## 📊 Eval Results'))
display(df[['id', 'domain', 'weighted_score', 'breadth', 'depth', 
            'recency', 'source_diversity', 'sources_count', 'expeditions', 'passed']]
        .set_index('id').style
        .background_gradient(subset=['weighted_score'], cmap='RdYlGn', vmin=0, vmax=10)
        .format({'weighted_score': '{:.2f}'}))

pass_rate = (df['passed'] == '✅').mean() * 100
avg_score = df['weighted_score'].mean()
print(f'\nPass Rate : {pass_rate:.0f}%')
print(f'Avg Score : {avg_score:.2f}/10')

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = f'results/eval_{timestamp}.json'

Path('results').mkdir(exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f'✅ Results saved to {output_path}')